# 05 · Evaluation metrics

Accuracy, per-class precision / recall, macro-F1 and confusion matrices (against the Final Gold v3 reference labels); evidence integrity (citation validity, future-data leakage, fiscal-period mismatches, reference-filing overlap); unsupported-conclusion rate; latency, tokens and estimated cost — grouped by approach, company, claim category and difficulty. Missing measurements stay missing (`NaN` with the count measured), never filled in.

In [ ]:
import os, sys, json
from dataclasses import asdict
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
print("research root:", ROOT)

In [ ]:
RUN_ID = None
EXPERIMENT_ID = "claimverif-main-v1"

In [ ]:
from engine.metrics import evaluate, classification_metrics
from engine.metrics import integrity_metrics, second_look_metrics
from engine.notebook import banner, pick_run, describe_run, load_gold_labels, reference_banner, trace_index, save_table, save_figure
from engine.schema import LABELS
run = pick_run(EXPERIMENT_ID, RUN_ID)
assert run is not None, "no run found -- execute notebook 03 first"
describe_run(run)
reference_banner()
preds, traces = run["predictions"], trace_index(run)
gold = load_gold_labels()
labelled = sum(1 for p in preds if p["claim_id"] in gold)
if not labelled:
    banner("None of this run's claims has a reference label (the reference covers the gold split only), so accuracy / "
           "precision / recall / macro-F1 are not computed. Gold-free measures (integrity, cost, latency) are shown.")
banner("Evidence precision / recall are not computed: the reference evidence quotes are not verbatim-verified "
       "(v3: 71/150 are corroborated table values, not quotes). Filing-level overlap with the reference is reported instead.", "info")

## By approach

In [ ]:
COLS = ["n_predictions", "n_labelled", "accuracy", "macro_f1", "evidence_precision", "evidence_recall", "n_evidence_scored",
        "decisive", "unsupported_conclusion_rate", "invalid_citations", "avg_latency_ms", "avg_input_tokens",
        "avg_output_tokens", "avg_llm_calls", "avg_cost_usd", "total_cost_usd", "n_cost", "is_mock"]
by_approach = pd.DataFrame(evaluate(preds, gold, traces, ("approach",)))
print("saved", save_table(by_approach, "metrics_by_approach", run))
by_approach[["approach"] + COLS]

### Figure: accuracy and macro-F1 per approach

Dots are point estimates; lines are 95% percentile bootstrap intervals over claims (5,000 resamples, fixed seed). Overlapping intervals do not by themselves mean no difference -- the paired tests in notebook 06 are the comparison.

In [ ]:
from engine import plots
from engine.stats import bootstrap_ci
plots.style()
RID = run["manifest"]["run_id"]
kind_of = {p["approach"]: p["approach_kind"] for p in preds}
summary = []
for a in sorted(kind_of):
    ps = [p for p in preds if p["approach"] == a and p["claim_id"] in gold]
    if not ps:
        continue
    ci = bootstrap_ci([gold[p["claim_id"]]["label"] for p in ps], [p["predicted_label"] for p in ps])
    summary.append({"approach": a, "kind": kind_of[a], "n": ci["n"],
                    "accuracy": ci["accuracy"]["point"], "acc_lo": ci["accuracy"]["ci_low"], "acc_hi": ci["accuracy"]["ci_high"],
                    "macro_f1": ci["macro_f1"]["point"], "f1_lo": ci["macro_f1"]["ci_low"], "f1_hi": ci["macro_f1"]["ci_high"]})
if summary:
    fig = plots.accuracy_summary(summary, f"Claim verification vs reference labels (n = {summary[0]['n']} claims)")
    print("saved", save_figure(fig, f"05_accuracy_{RID}", run)); plt.show()
    display(pd.DataFrame(summary).round(3))
    save_table(pd.DataFrame(summary), "accuracy_with_ci", run)
else:
    print("skipped: no labelled claims")

## Evidence integrity

- **citation_validity**: cited ids issued in the run, same company, filed by the cutoff, verbatim quote
- **future_data_leakage_rate**: runs that used any filing, passage or XBRL fact filed after the cutoff (must be 0)
- **incorrect_fiscal_period_rate**: runs with a numeric check whose model-extracted period or basis does not exist at the cutoff (a proxy; the reference has no period annotation)
- **reference_filing_hit_rate**: labelled runs citing at least one SEC filing the reference names (filing level)

In [ ]:
integ = pd.DataFrame(integrity_metrics(preds, traces, gold))
save_table(integ, "evidence_integrity", run)
integ

## Sensitivity: scoring decisive answers on INSUFFICIENT_EVIDENCE claims

| scoring | a decisive answer on an INSUFFICIENT_EVIDENCE claim counts as correct when |
|---|---|
| **strict** (primary) | never -- only an exact match with the reference |
| **adjudicated** | a person, blind to the approach, judged it ACCEPTABLE case by case |
| **lenient** | always (every decisive answer accepted) -- an upper bound |

Decisions come from `datasets/adjudication/` (`scripts/insufficient_adjudication.py`). Only claims whose reference is INSUFFICIENT_EVIDENCE can change; all other claims are scored strictly in every variant.

In [ ]:
from engine.reference import alternative_variants, load_alternatives, correct
from engine.stats import compare_correctness
from engine import plots
plots.style()
RID = run["manifest"]["run_id"]
variants = alternative_variants(RID)
if not variants:
    print("no adjudication recorded for this run -- see scripts/insufficient_adjudication.py")
else:
    ids = sorted({p["claim_id"] for p in preds} & set(gold))
    byc = {a: {p["claim_id"]: p for p in preds if p["approach"] == a} for a in sorted({p["approach"] for p in preds})}
    kind_of = {p["approach"]: p["approach_kind"] for p in preds}
    scorings = {"strict": None, **{v: load_alternatives(RID, v) for v in variants}}
    table, forest = [], []
    for name, alts in scorings.items():
        row = {"scoring": name}
        for a, m in byc.items():
            row[a] = sum(correct(m[i], gold, alts) for i in ids) / len(ids)
            if alts:
                row[f"{a} accepted"] = sum(1 for i in ids if alts.get((i, a)) == "ACCEPTABLE")
        table.append(row)
        for c in run["manifest"].get("comparisons") or []:
            if c["a"] in byc and c["b"] in byc:
                r = compare_correctness([correct(byc[c["a"]][i], gold, alts) for i in ids],
                                        [correct(byc[c["b"]][i], gold, alts) for i in ids])
                short = {"conventional_rag": "A1", "financial_verification": "A2", "contradiction_aware": "A3", "self_reflection": "SR"}
                forest.append({"label": f"{short[kind_of[c['a']]]} → {short[kind_of[c['b']]]}  ({name})",
                               "diff": r["accuracy_diff"]["diff_b_minus_a"], "lo": r["accuracy_diff"]["ci_low"],
                               "hi": r["accuracy_diff"]["ci_high"], "p": r["mcnemar"]["p_value_exact"], "_c": c["name"]})
    sens = pd.DataFrame(table)
    save_table(sens, "accuracy_by_scoring", run)
    display(sens.round(3))
    order = [c["name"] for c in run["manifest"].get("comparisons") or []]
    forest.sort(key=lambda r: (order.index(r["_c"]), list(scorings).index(r["label"].rsplit("(", 1)[1][:-1])))
    pd.DataFrame(forest).drop(columns="_c").pipe(lambda d: save_table(d, "paired_differences_by_scoring", run))
    fig = plots.paired_differences(forest, "Paired accuracy differences under three scorings of INSUFFICIENT_EVIDENCE claims")
    print("saved", save_figure(fig, f"05_scoring_sensitivity_{RID}", run)); plt.show()
    for v in variants:
        path = os.path.join("datasets", "adjudication", f"insufficient_alternatives__{RID}__{v}.jsonl")
        rows = [json.loads(line) for line in open(path)]
        print(f"{v}: {sum(r['decision'] == 'ACCEPTABLE' for r in rows)}/{len(rows)} answers accepted; "
              f"source {rows[0].get('source_workbook')}; adjudicator {rows[0].get('adjudicator')}")

## Agentic metrics (second look)

In [ ]:
slm = pd.DataFrame(second_look_metrics(preds, gold))
save_table(slm, "second_look_metrics", run)
slm

## By company, claim category and difficulty

In [ ]:
for keys in (("approach", "company"), ("approach", "claim_category"), ("approach", "difficulty")):
    t = pd.DataFrame(evaluate(preds, gold, traces, keys))
    save_table(t, "metrics_by_" + "_".join(keys), run)
    display(t[list(keys) + ["n_predictions", "n_labelled", "accuracy", "macro_f1", "unsupported_conclusion_rate", "avg_cost_usd"]])

In [ ]:
cat_rows = [{"approach": r["approach"], "kind": kind_of[r["approach"]], "claim_category": r["claim_category"],
             "accuracy": r["accuracy"], "n": r["n_labelled"]}
            for r in evaluate(preds, gold, traces, ("approach", "claim_category")) if r["n_labelled"]]
if cat_rows:
    fig = plots.category_accuracy(cat_rows, "Accuracy by claim category")
    print("saved", save_figure(fig, f"05_accuracy_by_category_{RID}", run)); plt.show()
else:
    print("skipped: no labelled claims")

## Per-class precision / recall and confusion matrices

In [ ]:
if labelled:
    cms = []
    for a in sorted(kind_of):
        ps = [p for p in preds if p["approach"] == a and p["claim_id"] in gold]
        m = classification_metrics([gold[p["claim_id"]]["label"] for p in ps], [p["predicted_label"] for p in ps])
        display(pd.DataFrame(m["per_class"]).T.assign(approach=a).round(3))
        cms.append((a, kind_of[a], m["confusion_matrix"]))
    fig = plots.confusion_grid(cms, "Confusion matrices (rows: reference label, columns: prediction)")
    print("saved", save_figure(fig, f"05_confusion_{RID}", run)); plt.show()
else:
    print("skipped: no labelled claims in this run")

## Self-check of the metric code (SYNTHETIC data, not research results)

Recomputes accuracy and macro-F1 by hand on synthetic labels and asserts the library agrees, so a reader can trust the tables above.

In [ ]:
from engine import synthetic
from engine.datasets import finalized_gold
sc = synthetic.synthetic_claims()
sa, sj = synthetic.synthetic_annotations(sc)
sgold = finalized_gold(sc, sa, sj)
spreds = synthetic.synthetic_predictions(sgold, {"A": 0.6, "B": 0.75})
res = {r["approach"]: r for r in evaluate(spreds, sgold, None, ("approach",))}
for a in ("A", "B"):
    ps = [p for p in spreds if p["approach"] == a]
    hand_acc = sum(p["predicted_label"] == sgold[p["claim_id"]]["label"] for p in ps) / len(ps)
    f1s = []
    for lab in LABELS:
        tp = sum(p["predicted_label"] == lab and sgold[p["claim_id"]]["label"] == lab for p in ps)
        fp = sum(p["predicted_label"] == lab and sgold[p["claim_id"]]["label"] != lab for p in ps)
        fn = sum(p["predicted_label"] != lab and sgold[p["claim_id"]]["label"] == lab for p in ps)
        if tp + fn:
            pr, rc = (tp / (tp + fp) if tp + fp else 0), tp / (tp + fn)
            f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0)
    assert abs(res[a]["accuracy"] - hand_acc) < 1e-12 and abs(res[a]["macro_f1"] - sum(f1s) / len(f1s)) < 1e-12
    print(f"synthetic {a}: accuracy {hand_acc:.3f}, macro-F1 {sum(f1s)/len(f1s):.3f} -- library matches hand computation")